# VCF Coordinate Remap + Flanking-Sequence Sanity Check

**Part A** — Use `rice_genome_query` utilities to produce a new VCF whose positions are in MSU7/IRGSP1 coordinates, so it aligns directly with `GCA_rice.fasta`.

**Part B** — Reproduce the flanking-sequence sanity check from `data_investigation.ipynb`, but using the remapped VCF + plain pysam/pyfaidx calls (no manual coordinate arithmetic).

In [1]:
import pandas as pd
import pysam
from pyfaidx import Fasta

from crop_embed.data.coords import (
    FASTA_PATH,
    VCF_PATH,
    FLANKING_PATH,
    build_msu6_to_msu7_map,
    remap_vcf_coordinates,
)

REMAPPED_VCF_PATH = "../rice_data/RiceDiversity_44K_Genotypes_PLINK/sativas413_msu7.vcf"

## Part A — Generate the MSU7-coordinate VCF

In [2]:
coord_map = build_msu6_to_msu7_map()
print(f"Coordinate map covers {len(coord_map):,} SNPs")
coord_map.head()

Coordinate map covers 42,755 SNPs


chr  pos_msu6
1    13147       14147
     32449       33449
     69990       70990
     73192       74192
     74969       75969
Name: pos_msu7, dtype: int64

In [3]:
counts = remap_vcf_coordinates(VCF_PATH, REMAPPED_VCF_PATH, coord_map)
counts

Remapped 35,690 / 36,901 records (1,211 dropped) → ../rice_data/RiceDiversity_44K_Genotypes_PLINK/sativas413_msu7.vcf


{'written': 35690, 'dropped': 1211, 'total': 36901}

In [4]:
# Diagnose the mismatch — inspect raw VCF records vs coord_map keys
vcf_debug = pysam.VariantFile(VCF_PATH)

print("First 5 VCF records:")
print(f"  {'chrom':>10}  {'rec.pos (0-based)':>18}  {'rec.pos+1':>10}  {'id':>12}")
for i, rec in enumerate(vcf_debug.fetch()):
    if i >= 5:
        break
    print(f"  {rec.chrom!r:>10}  {rec.pos:>18}  {rec.pos+1:>10}  {str(rec.id):>12}")
vcf_debug.close()

print("\nFirst 5 coord_map entries (index = (chr, pos_msu6), value = pos_msu7):")
print(coord_map.head())

print("\nIndex dtypes:", coord_map.index.dtypes.tolist())

# Check the first VCF record manually
vcf_debug = pysam.VariantFile(VCF_PATH)
rec = next(vcf_debug.fetch())
vcf_debug.close()

chrom_str = rec.chrom
chrom_int = int(chrom_str) if chrom_str.isdigit() else int(chrom_str.lstrip("chr"))
pos_1based = rec.pos + 1
key = (chrom_int, pos_1based)
print(f"\nFirst record key: {key}  →  coord_map.get: {coord_map.get(key)}")
print(f"Key types: {type(chrom_int)}, {type(pos_1based)}")
print(f"Index level types: {type(coord_map.index[0][0])}, {type(coord_map.index[0][1])}")

First 5 VCF records:
       chrom   rec.pos (0-based)   rec.pos+1            id
         '1'               13147       13148     id1000001
         '1'               73192       73193     id1000003
         '1'               74969       74970     id1000005
         '1'               75852       75853     id1000007
         '1'               75953       75954     id1000008

First 5 coord_map entries (index = (chr, pos_msu6), value = pos_msu7):
chr  pos_msu6
1    13147       14147
     32449       33449
     69990       70990
     73192       74192
     74969       75969
Name: pos_msu7, dtype: int64

Index dtypes: [dtype('int64'), dtype('int64')]

First record key: (1, 13148)  →  coord_map.get: None
Key types: <class 'int'>, <class 'int'>
Index level types: <class 'numpy.int64'>, <class 'numpy.int64'>


In [6]:
# Inspect the 1,211 dropped records
flanking_by_id  = pd.read_csv(FLANKING_PATH, sep="\t").set_index("snp_id")
flanking_by_pos = pd.read_csv(FLANKING_PATH, sep="\t").set_index(["chr", "pos"])

dropped = []
vcf_debug = pysam.VariantFile(VCF_PATH)
for rec in vcf_debug.fetch():
    chrom_int = int(rec.chrom) if rec.chrom.isdigit() else int(rec.chrom.lstrip("chr"))
    if coord_map.get((chrom_int, rec.pos)) is None:
        dropped.append({
            "snp_id":  rec.id,
            "chrom":   chrom_int,
            "pos_vcf": rec.pos,
            "in_flanking_by_id":  rec.id in flanking_by_id.index,
            "flanking_pos":       flanking_by_id.loc[rec.id, "pos"] if rec.id in flanking_by_id.index else None,
        })
vcf_debug.close()

dropped_df = pd.DataFrame(dropped)
print(f"Total dropped: {len(dropped_df)}")
print(f"\nID exists in flanking_seq (position mismatch): {dropped_df['in_flanking_by_id'].sum()}")
print(f"ID absent from flanking_seq entirely:          {(~dropped_df['in_flanking_by_id']).sum()}")

# For position mismatches, show how far off the positions are
mismatches = dropped_df[dropped_df["in_flanking_by_id"]].copy()
if len(mismatches):
    mismatches["pos_delta"] = mismatches["flanking_pos"] - mismatches["pos_vcf"]
    print("\nPosition delta distribution (flanking_pos - pos_vcf):")
    print(mismatches["pos_delta"].value_counts().head(10))

dropped_df.head(10)

Total dropped: 1211

ID exists in flanking_seq (position mismatch): 57
ID absent from flanking_seq entirely:          1154

Position delta distribution (flanking_pos - pos_vcf):
pos_delta
600.0    56
69.0      1
Name: count, dtype: int64


,snp_id,chrom,pos_vcf,in_flanking_by_id,flanking_pos
0,id1000139,1,347351,False,NaN
1,id1000759,1,851786,False,NaN
2,id1000911,1,1008219,False,NaN
3,id1001396,1,1836171,False,NaN
4,id1001464,1,1892584,False,NaN
5,id1001539,1,1973376,False,NaN
6,id1001849,1,2293711,False,NaN
7,id1001899,1,2370596,False,NaN
8,id1002060,1,2606283,False,NaN
9,id1002366,1,2975692,False,NaN


## Part B — Flanking-sequence sanity check

For each SNP in the remapped VCF, query the reference FASTA at `rec.pos` (now directly in MSU7 / 0-based coordinates) and compare the ±16 nt flanks against the stored `X5p_MSU6` / `X3p_MSU6` values.

A high match rate confirms the remap is correct.

In [7]:
# Load the stored flanking sequences, keyed by snp_id
flanking_seq = pd.read_csv(FLANKING_PATH, sep="\t").set_index("snp_id")

ref        = Fasta(FASTA_PATH)
chrom_name = {i + 1: name for i, name in enumerate(ref.keys())}
print("Chromosomes in FASTA:", chrom_name)

Chromosomes in FASTA: {1: 'ENA|AP014957|AP014957.1', 2: 'ENA|AP014958|AP014958.1', 3: 'ENA|AP014959|AP014959.1', 4: 'ENA|AP014960|AP014960.1', 5: 'ENA|AP014961|AP014961.1', 6: 'ENA|AP014962|AP014962.1', 7: 'ENA|AP014963|AP014963.1', 8: 'ENA|AP014964|AP014964.1', 9: 'ENA|AP014965|AP014965.1', 10: 'ENA|AP014966|AP014966.1', 11: 'ENA|AP014967|AP014967.1', 12: 'ENA|AP014968|AP014968.1'}


In [12]:
HALF_WINDOW = 16

results = []   # list of dicts: snp_id, chr, pos_msu7, left_match, right_match

vcf = pysam.VariantFile(REMAPPED_VCF_PATH)
for rec in vcf.fetch():
    snp_id = rec.id
    if snp_id not in flanking_seq.index:
        continue

    chrom_int = int(rec.chrom) if rec.chrom.isdigit() else int(rec.chrom.lstrip("chr"))
    seq_key   = chrom_name[chrom_int]

    # rec.pos is 0-based MSU7 — slice directly, no offset needed
    start  = max(0, rec.pos - HALF_WINDOW)
    window = ref[seq_key][start : rec.pos + HALF_WINDOW + 1].seq.upper()

    if len(window) != 2 * HALF_WINDOW + 1:
        continue   # near chromosome boundary

    stored = flanking_seq.loc[snp_id]
    stored_window = (stored["X5p_MSU6"] + stored["alleles"].split('/')[0] + stored["X3p_MSU6"]).upper()
    results.append({
        "snp_id":      snp_id,
        "chr":         chrom_int,
        "pos_msu7":    rec.pos + 1,   # 1-based for display
        "left_match":  window[:HALF_WINDOW]  == stored["X5p_MSU6"].upper(),
        "right_match": window[HALF_WINDOW+1:] == stored["X3p_MSU6"].upper(),
        "full_match":  window == stored_window,
    })

vcf.close()
results_df = pd.DataFrame(results)
print(f"SNPs checked: {len(results_df):,}")

SNPs checked: 35,689


In [13]:
both_match = results_df["left_match"] & results_df["right_match"]
full_match = results_df["full_match"]
n          = len(results_df)

print(f"Both flanks match : {both_match.sum():,} / {n:,}  ({100 * both_match.mean():.2f}%)")
print(f"Full match       : {full_match.sum():,} / {n:,}  ({100 * full_match.mean():.2f}%)")
print(f"Left flank only   : {(results_df['left_match'] & ~results_df['right_match']).sum():,}")
print(f"Right flank only  : {(~results_df['left_match'] & results_df['right_match']).sum():,}")
print(f"Neither           : {(~results_df['left_match'] & ~results_df['right_match']).sum():,}")

Both flanks match : 35,681 / 35,689  (99.98%)
Full match       : 35,680 / 35,689  (99.97%)
Left flank only   : 0
Right flank only  : 0
Neither           : 8


In [ ]:
# Per-chromosome breakdown
results_df.groupby("chr").apply(
    lambda g: pd.Series({
        "n_snps":       len(g),
        "both_match":   (g["left_match"] & g["right_match"]).sum(),
        "pct_match":    f"{100 * (g['left_match'] & g['right_match']).mean():.1f}%",
    }),
    include_groups=False,
)